# 02 — Range audit, missingness, and imputation files

Rebuild against `modelling_landmark_with_deltas.csv` after Phase 1.5 (`DXPOSINS`, BMI visit cleanup). Rules: `imputation_rules.md`.

**Does not overwrite** the Phase 1 CSVs. Writes **tree** copies only (NaNs kept; Part IV = 0; `dopamine_missing`).

There is **no dense CSV**. Full-cohort median/mode would leak test-set information. Sklearn RF / logistic regression must call `apply_dense_imputation(train, test)` **inside each Phase 3 seed split**.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'revision_work':
    ROOT = ROOT.parent
DATA = ROOT / 'revision_work' / 'data'

src = pd.read_csv(DATA / 'modelling_landmark_with_deltas.csv')
print(src.shape)
print('recurrent %', round(100 * src['falls_class'].eq(2).mean(), 1))
assert src['falls_class'].notna().all()
assert src['PATNO'].is_unique

(1040, 44)
recurrent % 9.7


## 1. Codebook / range audit

Illegal codes and leftover 101s should already be gone. This cell is the check, not a second cleaning pass.

In [2]:
RANGES = {
    'falls_class': (0, 2),
    'Dopaminergic therapy started for participant': (0, 1),
    'MoCA Total Score': (0, 30),
    'Freezing of gait (peak severity)': (0, 4),
    'lightheaded after standing': (0, 3),
    'fainted': (0, 3),
    'Total Depression Score': (0, 15),
    'MDS-UPDRS Part I Score': (0, 52),
    'BMI': (10, 60),
    'Daytime_Sleepiness': (0, 4),
    'Urinary_Problems': (0, 4),
    'Gait': (0, 4),
    'Postural_Stability': (0, 4),
    'Hoehn_And_Yahr_Stage': (0, 5),
    'Postural_hypotension': (0, 1),
    'MDS-UPDRS Part III Score': (0, 132),
    'MDS-UPDRS PartIV score': (0, 24),
    'Postural instability present at dx?': (0, 2),
    'Rigidity present at diagnosis?': (0, 2),
    'NP3GAIT_missing_due_to_101': (0, 1),
    'NP3PSTBL_missing_due_to_101': (0, 1),
    'NHY_missing_due_to_101': (0, 1),
}

rows = []
for col, (lo, hi) in RANGES.items():
    s = src[col]
    v = s.dropna()
    n_out = int(((v < lo) | (v > hi)).sum())
    leftover_101 = int((s == 101).sum()) if pd.api.types.is_numeric_dtype(s) else 0
    rows.append({
        'feature': col, 'min': v.min(), 'max': v.max(),
        'n_out_of_range': n_out, 'n_101': leftover_101, 'n_na': int(s.isna().sum()),
    })
audit = pd.DataFrame(rows)
print(audit.to_string(index=False))
assert audit['n_out_of_range'].eq(0).all(), audit[audit['n_out_of_range'] > 0]
assert audit['n_101'].eq(0).all()
print('\nBMI max (should be < 50 after visit cleanup):', round(src['BMI'].max(), 2))
print('|Delta BMI| > 10:', int((src['Delta BMI'].abs() > 10).sum()))
print('PI at dx Yes (DXPOSINS, expect ~92):', int(src['Postural instability present at dx?'].eq(1).sum()))

                                     feature       min       max  n_out_of_range  n_101  n_na
                                 falls_class  0.000000  2.000000               0      0     0
Dopaminergic therapy started for participant  0.000000  1.000000               0      0   202
                            MoCA Total Score  4.000000 30.000000               0      0     0
            Freezing of gait (peak severity)  0.000000  4.000000               0      0    63
                  lightheaded after standing  0.000000  3.000000               0      0     0
                                     fainted  0.000000  3.000000               0      0     0
                      Total Depression Score  0.000000 15.000000               0      0     0
                      MDS-UPDRS Part I Score  0.000000 17.000000               0      0     0
                                         BMI 14.915082 46.190818               0      0     0
                          Daytime_Sleepiness  0.000000  4.00

## 2. Missingness on the current extract

Most clinic scores are complete. Holes that remain are the ones `imputation_rules.md` is about.

In [3]:
META = ['PATNO', 'index_date', 'outcome_date', 'first_visit_date', 'history_months',
        'n_falls_visits', 'SEX', 'RACE', 'falls_raw', 'falls_class']
DELTA_COLS = [c for c in src.columns if c.startswith('Delta ')]
BASE_FEATS = [c for c in src.columns if c not in META and c not in DELTA_COLS]

miss = pd.DataFrame({
    'feature': BASE_FEATS + DELTA_COLS,
    'n_missing': [src[c].isna().sum() for c in BASE_FEATS + DELTA_COLS],
    'pct': [round(100 * src[c].isna().mean(), 1) for c in BASE_FEATS + DELTA_COLS],
}).query('n_missing > 0').sort_values('pct', ascending=False)
miss

,feature,n_missing,pct
27,Delta UPDRS IV,341,32.8
20,MDS-UPDRS PartIV score,226,21.7
33,Delta Neuro-QoL,211,20.3
4,Dopaminergic therapy started for participant,202,19.4
32,Delta BMI,143,13.8
28,Delta Depression,135,13.0
24,Delta MoCA,124,11.9
10,able_weighted_score,64,6.2
6,Freezing of gait (peak severity),63,6.1
26,Delta UPDRS III,57,5.5


In [4]:
dop = src['Dopaminergic therapy started for participant']
y = src['falls_class'].eq(2)
print('Dopamine vs recurrent %')
for label, m in [('No (0)', dop.eq(0)), ('Yes (1)', dop.eq(1)), ('Missing', dop.isna())]:
    sub = src[m]
    print(f'  {label:10} n={len(sub):4d}  rec={100*y[m].mean():5.1f}%  duration_med={sub["No_of_years"].median():.1f}')

p4 = src['MDS-UPDRS PartIV score']
print('\nPart IV vs recurrent %')
for label, m in [('Missing', p4.isna()), ('Observed 0', p4.eq(0)), ('Observed >0', p4.gt(0))]:
    sub = src[m]
    print(f'  {label:12} n={len(sub):4d}  rec={100*y[m].mean():5.1f}%  duration_med={sub["No_of_years"].median():.1f}')

Dopamine vs recurrent %
  No (0)     n= 153  rec=  3.3%  duration_med=2.4
  Yes (1)    n= 685  rec=  9.8%  duration_med=4.1
  Missing    n= 202  rec= 14.4%  duration_med=6.5

Part IV vs recurrent %
  Missing      n= 226  rec=  1.3%  duration_med=2.0
  Observed 0   n= 273  rec=  4.0%  duration_med=2.9
  Observed >0  n= 541  rec= 16.1%  duration_med=9.6


## 3. Clinical fills (both backends)

- `dopamine_missing` indicator; **do not** fill dopamine with 0
- Part IV missing → 0
- Hypotension (5) and PI-at-dx (1) → training/full **mode** (tiny holes; 2=Unknown stays 2)

In [5]:
DOP = 'Dopaminergic therapy started for participant'
P4 = 'MDS-UPDRS PartIV score'
FOG = 'Freezing of gait (peak severity)'
ABLE = 'able_weighted_score'
HYPO = 'Postural_hypotension'
PI = 'Postural instability present at dx?'

TINY = [HYPO, PI]


def clinical_fills(df):
    out = df.copy()
    out['dopamine_missing'] = out[DOP].isna().astype(int)
    out[P4] = out[P4].fillna(0)
    return out


def _mode(series):
    m = series.mode(dropna=True)
    return m.iloc[0] if len(m) else np.nan


def fill_tiny(df, ref):
    out = df.copy()
    for c in TINY:
        out[c] = out[c].fillna(_mode(ref[c]))
    return out


base = clinical_fills(src)
base = fill_tiny(base, src)
assert base[P4].notna().all()
print('dopamine_missing n', int(base['dopamine_missing'].sum()))
print('Part IV still NaN', int(base[P4].isna().sum()))
print('tiny still NaN', {c: int(base[c].isna().sum()) for c in TINY})

dopamine_missing n 202
Part IV still NaN 0
tiny still NaN {'Postural_hypotension': 0, 'Postural instability present at dx?': 0}


## 4. Tree files (NaN kept)

XGBoost / LightGBM. Dopamine, FOG, Neuro-QoL, deltas stay missing. Part IV is already 0.

In [6]:
tree = base.copy()
# dopamine stays NaN on purpose
assert tree.loc[tree['dopamine_missing'].eq(1), DOP].isna().all()

tree_with = tree
tree_no = tree.drop(columns=DELTA_COLS)

tree_with.to_csv(DATA / 'modelling_landmark_with_deltas_trees.csv', index=False)
tree_no.to_csv(DATA / 'modelling_landmark_trees.csv', index=False)
print('trees with deltas', tree_with.shape, 'NaNs in features',
      int(tree_with.drop(columns=META).isna().sum().sum()))
print('trees no deltas  ', tree_no.shape)

trees with deltas (1040, 45) NaNs in features 1567
trees no deltas   (1040, 35)


## 5. Dense models — train-only, inside the split (no CSV)

Sklearn RandomForest cannot take NaN. Do **not** write a filled table: median/mode on all 1040 leaks the test fold.

Phase 3: split first, then `apply_dense_imputation(train, test)`. Fit on train, apply to test. Every seed gets its own fills.

Deltas: 0 + `{col}_missing` flag. Bare 0 would mean “stable.”

In [7]:
def apply_dense_imputation(train, test):
    """Train-only median/mode. Call after the seed split, never on the full 1040.

    `train` / `test` should already have clinical fills (Part IV = 0, dopamine_missing)
    as in the tree CSVs. Statistical fills (dopamine/FOG mode, Neuro-QoL median,
    tiny holes, delta 0 + flag) are computed on train and applied to test.
    """
    tr, te = train.copy(), test.copy()
    for c in TINY:
        fill = _mode(tr[c])
        tr[c] = tr[c].fillna(fill)
        te[c] = te[c].fillna(fill)
    if 'dopamine_missing' not in tr.columns:
        tr['dopamine_missing'] = tr[DOP].isna().astype(int)
        te['dopamine_missing'] = te[DOP].isna().astype(int)
    dop_mode = _mode(tr[DOP])
    tr[DOP] = tr[DOP].fillna(dop_mode)
    te[DOP] = te[DOP].fillna(dop_mode)
    fog_mode = _mode(tr[FOG])
    tr[FOG] = tr[FOG].fillna(fog_mode)
    te[FOG] = te[FOG].fillna(fog_mode)
    able_med = tr[ABLE].median()
    tr[ABLE] = tr[ABLE].fillna(able_med)
    te[ABLE] = te[ABLE].fillna(able_med)
    for c in [c for c in tr.columns if c.startswith('Delta ')]:
        flag = c + '_missing'
        tr[flag] = tr[c].isna().astype(int)
        te[flag] = te[c].isna().astype(int)
        tr[c] = tr[c].fillna(0)
        te[c] = te[c].fillna(0)
    if P4 in tr.columns:
        tr[P4] = tr[P4].fillna(0)
        te[P4] = te[P4].fillna(0)
    return tr, te


# Smoke test: a 70/30 split must come back with no feature NaNs, without writing a CSV.
from sklearn.model_selection import train_test_split

tr0, te0 = train_test_split(tree, test_size=0.3, stratify=tree['falls_class'], random_state=0)
tr1, te1 = apply_dense_imputation(tr0, te0)
feat = [c for c in tr1.columns if c not in META]
print('train n', len(tr1), 'test n', len(te1))
print('feature NaNs after fill: train', int(tr1[feat].isna().sum().sum()),
      'test', int(te1[feat].isna().sum().sum()))
print('dopamine mode used', _mode(tr0[DOP]), '| FOG mode', _mode(tr0[FOG]),
      '| able median', round(tr0[ABLE].median(), 3))
assert tr1[feat].isna().sum().sum() == 0
assert te1[feat].isna().sum().sum() == 0
print('No dense CSV written. Phase 3 must call apply_dense_imputation inside each seed.')


train n 728 test n 312
feature NaNs after fill: train 0 test 0
dopamine mode used 1.0 | FOG mode 0.0 | able median 1.601
No dense CSV written. Phase 3 must call apply_dense_imputation inside each seed.


## 6. What was written

| File | Use |
|------|-----|
| `modelling_landmark.csv` / `_with_deltas.csv` | Phase 1 source of truth (NaNs, no fills) |
| `modelling_landmark_trees.csv` / `_with_deltas_trees.csv` | XGBoost etc. Load these. |
| *(no `*_dense.csv`)* | Sklearn RF: `apply_dense_imputation` after each split |
